In [0]:
import uuid

In [0]:
storage_account = "annagaplanyandatabricks"
container = "pdb-data"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope="adls-scope", key="storage-key")
)

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/pdb_lab"
system_path = f"{base_path}/system"

In [0]:
def log_event(run_id, stage, table_name, status, record_count=None, message=None):
    log_id = str(uuid.uuid4())
    msg = (message or "").replace("'", "''")
    run_id_sql = "NULL" if run_id is None else f"'{str(run_id)}'"

    spark.sql(f"""
        INSERT INTO pdb_lab.pipeline_logs
        SELECT
            '{log_id}',
            {run_id_sql},
            '{stage}',
            '{table_name}',
            '{status}',
            {record_count if record_count is not None else "NULL"},
            '{msg}',
            current_timestamp()
    """)


In [0]:
run_id = None

tables = [
    "pdb_lab.silver_entity_mv",
    "pdb_lab.silver_exptl_mv",
    "pdb_lab.silver_entity_poly_seq_mv",
    "pdb_lab.silver_chem_comp_mv",
    "pdb_lab.silver_pdbx_database_pdb_obs_spr_mv"
]

log_event(run_id, "silver", "pipeline", "STARTED")

try:
    for table in tables:
        try:
            log_event(run_id, "silver", table, "STARTED")
            count = spark.table(table).count()
            log_event(run_id, "silver", table, "SUCCESS", record_count=count)
        except Exception as e:
            log_event(run_id, "silver", table, "FAILED", message=str(e))
            raise

    log_event(run_id, "silver", "pipeline", "SUCCESS")

except Exception as e:
    log_event(run_id, "silver", "pipeline", "FAILED", message=str(e))
    raise